In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score,
                            recall_score, f1_score, roc_auc_score,
                            classification_report)
from sklearn.preprocessing import StandardScaler
import pickle

processed_path = Path('../data/processed')

X_train = pd.read_csv(processed_path / 'X_train.csv')
X_test = pd.read_csv(processed_path / 'X_test.csv')
y_train = pd.read_csv(processed_path / 'y_train.csv').squeeze()
y_test = pd.read_csv(processed_path / 'y_test.csv').squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

X_train: (3539, 41)
X_test: (885, 41)


In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

print("Features scaled")

Features scaled


In [9]:
models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        class_weight='balanced', n_estimators=100, random_state=42
    ),
    'XGBoost': XGBClassifier(
        scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
        n_estimators=100, random_state=42, eval_metric='logloss'
    )
}

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    X = X_train_scaled if name == 'Logistic Regression' else X_train

    auc = cross_val_score(model, X, y_train, cv=cv, scoring='roc_auc').mean()
    f1 = cross_val_score(model, X, y_train, cv=cv, scoring='f1').mean()
    recall = cross_val_score(model, X, y_train, cv=cv, scoring='recall').mean()

    results[name] = {'ROC-AUC': round(auc, 4),
                     'F1': round(f1, 4),
                      'Recall': round(recall, 4)}
    print(f"{name} done")

print("\n--- Cross-Validation Results ---")
print(pd.DataFrame(results).T)

Logistic Regression done
Random Forest done
XGBoost done

--- Cross-Validation Results ---
                     ROC-AUC      F1  Recall
Logistic Regression   0.9148  0.7883  0.8092
Random Forest         0.9186  0.7878  0.7247
XGBoost               0.9142  0.7838  0.7616


In [12]:
best_model = XGBClassifier(
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    n_estimators=100, random_state=42, eval_metric='logloss'
)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("--- Test Set performance --")
print(classification_report(y_test, y_pred, target_names=['No Dropoud', 'Dropout']))
print(f"ROC-AUC: {roc_auc_score(y_test, best_model.predict_proba(X_test)[:,1]):.4f}")

models_path = Path('../models')
with open(models_path / 'trained_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open(models_path / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("\n Model and scaler saved to models/")

--- Test Set performance --
              precision    recall  f1-score   support

  No Dropoud       0.91      0.93      0.92       601
     Dropout       0.84      0.82      0.83       284

    accuracy                           0.89       885
   macro avg       0.88      0.87      0.88       885
weighted avg       0.89      0.89      0.89       885

ROC-AUC: 0.9292

 Model and scaler saved to models/


In [13]:
from sklearn.linear_model import LogisticRegression
import pickle

# Train Logistic Regression on full training set
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Evaluate on test set
y_pred_lr = lr_model.predict(X_test_scaled)

print("--- Logistic Regression Test Set Performance ---")
print(classification_report(y_test, y_pred_lr, target_names=['No Dropout', 'Dropout']))
print(f"ROC-AUC: {roc_auc_score(y_test, lr_model.predict_proba(X_test_scaled)[:,1]):.4f}")

--- Logistic Regression Test Set Performance ---
              precision    recall  f1-score   support

  No Dropout       0.92      0.89      0.90       601
     Dropout       0.78      0.83      0.81       284

    accuracy                           0.87       885
   macro avg       0.85      0.86      0.86       885
weighted avg       0.87      0.87      0.87       885

ROC-AUC: 0.9267
